<a href="https://colab.research.google.com/github/ssjeson412/BDS-M1/blob/M1/recommender_systems/M1_Recap_Recommender_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
# Importing necessary libraries.
import pandas as pd
import scipy.sparse as ss
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_distances

In [27]:
df_trips = pd.read_csv('https://sds-aau.github.io/SDS-master/M1/data/trips.csv')

In [28]:
df_trips.head()

,Unnamed: 0,username,country,country_code,country_slug,date_end,date_start,latitude,longitude,place,place_slug
0,0,@lewellenmichael,Mexico,MX,mexico,2018-06-15,2018-06-04,21,-101,Guanajuato,mexico
1,1,@lewellenmichael,Mexico,MX,mexico,2018-06-03,2018-05-31,19,-99,Mexico City,mexico-city-mexico
2,2,@lewellenmichael,Mexico,MX,mexico,2017-11-05,2017-11-01,21,-86,Cancun,cancun-mexico
3,3,@lewellenmichael,Jordan,JO,jordan,2017-08-07,2017-07-24,31,35,Amman,amman-jordan
4,4,@waylandchin,China,CN,china,2017-03-18,2017-02-17,40,122,Yingkou,china


In [42]:
# Creating user-place matrix using df_trips
trips_matrix = pd.crosstab(df_trips['username'], df_trips['place_slug'])
display(trips_matrix.head())

place_slug,11000,180-40,45230,630-72,671-03,847-00,a-coruna-spain,aachen-germany,aalborg-denmark,aalesund-norway,...,york-united-kingdom,zadar-croatia,zagreb-croatia,zambia,zhangjiakou-china,zhengzhou-china,zhuhai-china,zimbabwe,zurich-switzerland,zwolle-netherlands
username,,,,,,,,,,,,,,,,,,,,,
@01aniqe,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
@0chucha0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
@10kjuan,0,0,0,0,0,0,0,0,0,0,...,0,0,3,0,0,0,0,0,0,0
@123456,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
@1ikigai,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [29]:
###################################################
#   Step 1: Label Encoding and Matrix Creation    #
###################################################
# Initialize label encoders
le_user = LabelEncoder()
le_place = LabelEncoder()
# Label encode usernames and place slugs
df_trips['username_id'] = le_user.fit_transform(df_trips['username'])
df_trips['place_slug_id'] = le_place.fit_transform(df_trips['place_slug'])

# Create a sparse matrix

# Create the sparse matrix using the 'username_id' and 'place_slug_id' columns as indices
matrix = pd.crosstab(df_trips['username_id'], df_trips['place_slug_id'])

###################################################
#    Step 2: Perform Dimensionality Reduction     #
###################################################
svd = TruncatedSVD(n_components=5, n_iter=7, random_state=42)
#These latent features could potentially capture characteristics like how popular a place is
matrix_places = svd.fit_transform(matrix.T)

###################################################
#    Step 3: Calculate The Similarity Matrix      #
###################################################
cosine_distance_matrix_places = cosine_distances(matrix_places)

In [30]:
cosine_distance_matrix_places.shape

(961, 961)

In [31]:
cosine_distance_matrix_places

array([[0.        , 0.93737257, 0.94042055, ..., 0.55101577, 0.94479183,
        0.66700793],
       [0.93737257, 0.        , 0.2219977 , ..., 0.55865144, 1.01683665,
        0.55894041],
       [0.94042055, 0.2219977 , 0.        , ..., 0.42547908, 0.90915149,
        0.12785817],
       ...,
       [0.55101577, 0.55865144, 0.42547908, ..., 0.        , 0.35913659,
        0.36126287],
       [0.94479183, 1.01683665, 0.90915149, ..., 0.35913659, 0.        ,
        0.89817704],
       [0.66700793, 0.55894041, 0.12785817, ..., 0.36126287, 0.89817704,
        0.        ]])

In [32]:
df_trips = pd.read_csv('https://sds-aau.github.io/SDS-master/M1/data/trips.csv')
df_trips.shape # (46510, 14)

(46510, 11)

In [33]:
le_users, le_items = LabelEncoder(), LabelEncoder()
df_trips['username_id'] = le_users.fit_transform(df_trips['username'])
df_trips['place_slug_id'] = le_items.fit_transform(df_trips['place_slug'])
matrix_user_place = pd.crosstab(df_trips['username_id'], df_trips['place_slug_id'])
matrix_user_place.shape # (2871, 961)


(2871, 961)

In [34]:
svd = TruncatedSVD(n_components=5, n_iter=7, random_state=42)
matrix_user_place_dr = svd.fit_transform(matrix_user_place)
matrix_user_place_dr.shape # (2871, 5)

(2871, 5)

In [35]:
D = cosine_distances(matrix_user_place_dr) # (2871, 2871)
i = le_users.transform(['@cw'])[0] # 555
np.argsort(D[i, :])[:5] # [555, 2644, 153, 1371, 702]
le_users.inverse_transform(np.argsort(D[i, :])[:5])

array(['@cw', '@travpreneur', '@andrea', '@katehuentelman', '@dpashley'],
      dtype=object)

In [36]:
mine = df_trips[df_trips.username_id == 555]['place_slug_id'].unique()
theirs = df_trips[df_trips.username_id == 2644]['place_slug_id'].unique()
le_items.inverse_transform(theirs[~np.isin(theirs, mine)])[:10]

array(['pune-india', 'manchester-united-kingdom',
       'san-diego-ca-united-states', 'san-jose-ca-united-states',
       'amsterdam-netherlands', 'birmingham-united-kingdom',
       'mumbai-india', 'victoria-seychelles', 'johannesburg-south-africa',
       'delhi-india'], dtype=object)

In [38]:
matrix_place_user = matrix_user_place.T # (961, 2871)
matrix_place_user_dr = svd.fit_transform(matrix_place_user) # (961, 5)
Dp = cosine_distances(matrix_place_user_dr) # (961, 961)

In [40]:
j = le_items.transform(['paris-france'])[0] # 626
np.argsort(Dp[j, :])[:5] # [626, 735, 12, 172, 856]
le_items.inverse_transform(np.argsort(Dp[j, :])[:5])

array(['paris-france', 'france', 'grenoble-france', 'gothenburg-sweden',
       'malta'], dtype=object)

In [41]:
# Constructing the user-to-place matrix
user_place_matrix = pd.crosstab(df_trips['username'], df_trips['place_slug'])
print("Matrix Shape:", user_place_matrix.shape)
user_place_matrix.head()

Matrix Shape: (2869, 960)


place_slug,11000,180-40,45230,630-72,671-03,847-00,a-coruna-spain,aachen-germany,aalborg-denmark,aalesund-norway,...,york-united-kingdom,zadar-croatia,zagreb-croatia,zambia,zhangjiakou-china,zhengzhou-china,zhuhai-china,zimbabwe,zurich-switzerland,zwolle-netherlands
username,,,,,,,,,,,,,,,,,,,,,
@01aniqe,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
@0chucha0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
@10kjuan,0,0,0,0,0,0,0,0,0,0,...,0,0,3,0,0,0,0,0,0,0
@123456,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
@1ikigai,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
